# LangGraph RAG Agent: Retrieve, Grade, Rewrite, Answer

This notebook builds a small but realistic **RAG agent with LangGraph**.

The knowledge base is a small inline mock corpus about classical music composers.

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain why RAG agents benefit from LangGraph.
2. Define state for a RAG workflow.
3. Build nodes for retrieval, grading, rewriting, answering, and fallback.
4. Use conditional edges to route execution.
5. Add a retry loop.
6. Draw and inspect the graph.
7. Run a small RAG agent end to end.

## Why LangGraph Instead of Just LangChain?

A simple LangChain chain is often linear:

```text
question -> retrieve -> prompt -> LLM -> answer
```

That is good for basic RAG.

But agentic RAG often needs control flow:

```text
question
   |
   v
retrieve documents
   |
   v
are documents relevant?
   | yes
   v
generate answer -> END

   | no
   v
rewrite question
   |
   v
retrieve again
```

LangGraph is useful because this workflow has:

- branching
- retries
- state shared across steps
- debug-friendly intermediate outputs
- a clear graph structure

## Target Graph

We will build this graph:

```text
START
  |
  v
initialize
  |
  v
retrieve
  |
  v
grade_context
  |
  +-- relevant ---------> generate_answer -> END
  |
  +-- weak + retries ---> rewrite_question -> retrieve
  |
  +-- weak + no retries -> fallback_answer -> END
```

This is intentionally more complex than a linear chain.

## Setup

This notebook uses:

- `langgraph`
- `langchain-openai` if an OpenAI API key is available
- `scikit-learn` for a tiny local TF-IDF retriever
- `python-dotenv` for optional `.env` loading

The notebook can run without an API key. If no `OPENAI_API_KEY` is found, the answer node uses a simple extractive fallback instead of calling an LLM.

In [1]:
# Run once if needed, then restart the kernel.

%pip install -U "langgraph" "langchain-core" "scikit-learn" "python-dotenv"


  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached ormsgpack-1.12.2-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
  Using cached orjson-3.12.0-cp312-cp312-win_amd64.whl.metadata (43 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/572.1 kB ? eta -:--:--
   ------------------ --------------------- 262.1/572.1 kB ? eta -:--:--
   ---------------------------------------- 572.1/572.1 kB 5.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.3 MB 9.0 MB/s eta 0:00:01
   ---------------------- ----------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
docling 2.31.2 requires typer<0.16.0,>=0.12.5, but you have typer 0.27.2 which is incompatible.
docling-core 2.30.0 requires typer<0.16.0,>=0.12.5, but you have typer 0.27.2 which is incompatible.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.6.5 which is incompatible.
langchain 0.3.25 requires langsmith<0.4,>=0.1.17, but you have langsmith 0.14.0 which is incompatible.
langchain-ollama 0.3.2 requires langchain-core<1.0.0,>=0.3.52, but you have langchain-core 1.6.5 which is incompatible.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 1.6.5 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports and Configuration

We keep the model optional so the notebook remains runnable in classrooms without API access.

In [2]:
import importlib
import os
import re
import sys
import types
from importlib.metadata import version
from operator import add
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from IPython.display import Image, Markdown, display
from langgraph.graph import END, START, StateGraph
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv(override=False)

# ---------------------------------------------------------------------------
# Route all LLM calls through the LiteLLM proxy — no OpenAI key required.
# We reload utils.open_ai to pick up the latest version, then patch
# langchain_openai so the generate_answer node works completely unchanged.
# ---------------------------------------------------------------------------
import utils.open_ai as _openai_module
importlib.reload(_openai_module)
from utils.open_ai import ChatOpenAI

_fake_lc_openai = types.ModuleType("langchain_openai")
_fake_lc_openai.ChatOpenAI = ChatOpenAI
sys.modules["langchain_openai"] = _fake_lc_openai

MODEL = os.environ.get("MODEL")
USE_LLM = True   # always True — LiteLLM proxy is always available

print("langgraph:", version("langgraph"))
print("scikit-learn:", version("scikit-learn"))
print("Use LLM:", USE_LLM)
print("Model:", MODEL)


langgraph: 1.2.12
scikit-learn: 1.9.1
Use LLM: True
Model: gpt-4.1-mini


## Create a Tiny Knowledge Base

In production, this data could come from:

- PDFs
- Confluence pages
- SharePoint
- product documentation
- internal knowledge bases

We use a tiny inline corpus so everyone can see exactly what retrieval is searching.

In [3]:
DOCUMENTS = [
    {
        "source": "doc_1_chopin_style",
        "text": (
            "Frederic Chopin was a Polish composer and pianist of the Romantic era. "
            "He is especially known for expressive piano music, lyrical melodies, rubato, "
            "nocturnes, mazurkas, polonaises, and etudes."
        ),
    },
    {
        "source": "doc_2_chopin_paris",
        "text": (
            "Chopin spent much of his adult life in Paris, where he became connected with "
            "salon culture and many artists of the nineteenth century. His performances were "
            "often intimate rather than large public concerts."
        ),
    },
    {
        "source": "doc_3_bach",
        "text": (
            "Johann Sebastian Bach was a German Baroque composer. He is known for counterpoint, "
            "fugues, chorales, sacred music, and works such as The Well-Tempered Clavier."
        ),
    },
    {
        "source": "doc_4_beethoven",
        "text": (
            "Ludwig van Beethoven helped bridge the Classical and Romantic eras. His symphonies, "
            "piano sonatas, and string quartets expanded musical form and emotional intensity."
        ),
    },
    {
        "source": "doc_5_debussy",
        "text": (
            "Claude Debussy was a French composer associated with impressionist colors, unusual "
            "scales, atmosphere, and works such as Prelude to the Afternoon of a Faun."
        ),
    },
]

for doc in DOCUMENTS:
    print(doc["source"], "->", doc["text"][:90] + "...")

doc_1_chopin_style -> Frederic Chopin was a Polish composer and pianist of the Romantic era. He is especially kn...
doc_2_chopin_paris -> Chopin spent much of his adult life in Paris, where he became connected with salon culture...
doc_3_bach -> Johann Sebastian Bach was a German Baroque composer. He is known for counterpoint, fugues,...
doc_4_beethoven -> Ludwig van Beethoven helped bridge the Classical and Romantic eras. His symphonies, piano ...
doc_5_debussy -> Claude Debussy was a French composer associated with impressionist colors, unusual scales,...


## Build a Local Retriever

To keep the notebook simple, we use TF-IDF retrieval.

This is not as powerful as embeddings, as we know.

```text
question -> TF-IDF similarity -> top matching documents
```

In [4]:
corpus_texts = [doc["text"] for doc in DOCUMENTS]
vectorizer = TfidfVectorizer(stop_words="english")
document_matrix = vectorizer.fit_transform(corpus_texts)


def retrieve_documents(query: str, k: int = 3) -> list[dict]:
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, document_matrix).flatten()
    ranked_indexes = similarities.argsort()[::-1][:k]

    results = []
    for index in ranked_indexes:
        doc = DOCUMENTS[index]
        results.append(
            {
                "source": doc["source"],
                "text": doc["text"],
                "score": float(similarities[index]),
            }
        )

    return results


retrieve_documents("What makes Chopin's piano music distinctive?", k=2)

[{'source': 'doc_1_chopin_style',
  'text': 'Frederic Chopin was a Polish composer and pianist of the Romantic era. He is especially known for expressive piano music, lyrical melodies, rubato, nocturnes, mazurkas, polonaises, and etudes.',
  'score': 0.3419201250226246},
 {'source': 'doc_3_bach',
  'text': 'Johann Sebastian Bach was a German Baroque composer. He is known for counterpoint, fugues, chorales, sacred music, and works such as The Well-Tempered Clavier.',
  'score': 0.12724157167165318}]

## Define Graph State

State is the shared memory of the graph.

Every node receives state and returns updates to state.

We also keep a `trace` list we can see which nodes ran.

In [5]:
class RAGState(TypedDict, total=False):
    original_question: str
    question: str
    attempts: int
    max_attempts: int
    retrieved_docs: list[dict]
    context_is_relevant: bool
    answer: str
    route_reason: str
    trace: Annotated[list[str], add]

## Helper Functions

These small helpers keep the graph nodes readable.

In [6]:
STOP_WORDS = {
    "the", "a", "an", "and", "or", "to", "of", "in", "is", "are", "was", "were",
    "what", "which", "who", "when", "where", "why", "how", "tell", "me", "about",
}


def keywords(text: str) -> set[str]:
    words = re.findall(r"[a-zA-Z']+", text.lower())
    return {word.strip("'") for word in words if len(word) > 2 and word not in STOP_WORDS}


def format_docs(docs: list[dict]) -> str:
    formatted = []
    for doc in docs:
        formatted.append(
            f'Source: {doc["source"]} | score={doc["score"]:.3f}\n{doc["text"]}'
        )
    return "\n\n".join(formatted)

## Node 1: Initialize

This node normalizes the incoming state.

It makes sure the graph has:

- an original question
- a current question
- an attempt counter
- a maximum number of retries

In [7]:
def initialize(state: RAGState) -> dict:
    question = state["question"]
    return {
        "original_question": state.get("original_question", question),
        "question": question,
        "attempts": state.get("attempts", 0),
        "max_attempts": state.get("max_attempts", 2),
        "trace": ["initialize"],
    }

## Node 2: Retrieve

This node searches the local knowledge base.

In [8]:
def retrieve(state: RAGState) -> dict:
    docs = retrieve_documents(state["question"], k=3)
    return {
        "retrieved_docs": docs,
        "trace": [f'retrieve: "{state["question"]}"'],
    }

## Node 3: Grade Context

This node decides whether the retrieved documents look useful enough.

For teaching, we use a deterministic rule:

- at least one retrieved document has a decent score
- and the original question shares at least one meaningful keyword with the retrieved documents

In production, this could be an LLM grader or a more advanced ranking model.

In [9]:
def grade_context(state: RAGState) -> dict:
    question_terms = keywords(state.get("original_question", state["question"]))
    docs = state["retrieved_docs"]

    best_score = max((doc["score"] for doc in docs), default=0.0)
    combined_doc_terms = set()
    for doc in docs:
        combined_doc_terms.update(keywords(doc["text"]))

    overlap = question_terms.intersection(combined_doc_terms)
    is_relevant = best_score >= 0.25 and len(overlap) >= 1

    reason = (
        f"best_score={best_score:.3f}, "
        f"keyword_overlap={sorted(overlap)}"
    )

    return {
        "context_is_relevant": is_relevant,
        "route_reason": reason,
        "trace": [f"grade_context: {is_relevant} ({reason})"],
    }

## Conditional Router

This function chooses the next node after grading.

```text
relevant context -> generate_answer
weak context + attempts left -> rewrite_question
weak context + no attempts left -> fallback_answer
```

In [10]:
def route_after_grading(state: RAGState) -> str:
    if state["context_is_relevant"]:
        return "generate"

    if state["attempts"] < state["max_attempts"]:
        return "rewrite"

    return "fallback"

## Node 4: Rewrite Question

If retrieval is weak, the graph rewrites the question and tries again.

This is a simple deterministic rewrite. In a production agent, an LLM could rewrite the query.

In [11]:
def rewrite_question(state: RAGState) -> dict:
    attempts = state["attempts"] + 1
    rewritten = (
        f'{state["original_question"]} '
        "classical music composer style biography works Romantic Baroque"
    )

    return {
        "question": rewritten,
        "attempts": attempts,
        "trace": [f"rewrite_question: attempt {attempts}"],
    }

## Node 5: Generate Answer

If `OPENAI_API_KEY` is available, this node calls an LLM through LangChain.

If no key is available, it returns a simple extractive answer from the retrieved context. So you can run this in the TCS computer.

In [12]:
def generate_answer(state: RAGState) -> dict:
    context = format_docs(state["retrieved_docs"])
    question = state["original_question"]

    if USE_LLM:
        from langchain_core.messages import HumanMessage, SystemMessage
        from langchain_openai import ChatOpenAI

        llm = ChatOpenAI(model=MODEL, temperature=0)
        messages = [
            SystemMessage(
                content=(
                    "You are a careful RAG assistant. Answer only from the provided context. "
                    "Cite source ids like [doc_1_chopin_style]. If the context is insufficient, say so."
                )
            ),
            HumanMessage(
                content=(
                    f"Question:\n{question}\n\n"
                    f"Context:\n{context}\n\n"
                    "Answer in one short paragraph."
                )
            ),
        ]
        response = llm.invoke(messages)
        answer = response.content
    else:
        best_doc = state["retrieved_docs"][0]
        answer = (
            "LLM fallback answer: "
            f'Based on [{best_doc["source"]}], {best_doc["text"]}'
        )

    return {
        "answer": answer,
        "trace": ["generate_answer"],
    }

## Node 6: Fallback Answer

If retrieval stays weak after retries, the graph stops safely instead of hallucinating.

In [13]:
def fallback_answer(state: RAGState) -> dict:
    return {
        "answer": (
            "I could not find enough relevant information in the local knowledge base "
            "to answer confidently."
        ),
        "trace": ["fallback_answer"],
    }

## Build the LangGraph App

This is where LangGraph becomes useful.

We define nodes, normal edges, conditional edges, and a retry loop.

In [14]:
graph_builder = StateGraph(RAGState)

graph_builder.add_node("initialize", initialize)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("grade_context", grade_context)
graph_builder.add_node("rewrite_question", rewrite_question)
graph_builder.add_node("generate_answer", generate_answer)
graph_builder.add_node("fallback_answer", fallback_answer)

graph_builder.add_edge(START, "initialize")
graph_builder.add_edge("initialize", "retrieve")
graph_builder.add_edge("retrieve", "grade_context")

graph_builder.add_conditional_edges(
    "grade_context",
    route_after_grading,
    {
        "generate": "generate_answer",
        "rewrite": "rewrite_question",
        "fallback": "fallback_answer",
    },
)

graph_builder.add_edge("rewrite_question", "retrieve")
graph_builder.add_edge("generate_answer", END)
graph_builder.add_edge("fallback_answer", END)

rag_graph = graph_builder.compile()

print("RAG graph compiled.")

RAG graph compiled.


## Draw the Graph

The PNG renderer looks nicest when available. If it cannot render in your environment, the notebook falls back to Mermaid text.

In [15]:
graph_visual = rag_graph.get_graph()
mermaid_graph = graph_visual.draw_mermaid()

try:
    png_bytes = graph_visual.draw_mermaid_png()
    display(Image(png_bytes))
except Exception as exc:
    print("PNG rendering is not available in this environment.")
    print("Showing Mermaid text instead.")
    print("Reason:", exc)

display(Markdown(f"```mermaid\n{mermaid_graph}\n```"))

PNG rendering is not available in this environment.
Showing Mermaid text instead.
Reason: Failed to reach https://mermaid.ink API while trying to render your graph after 1 retries. To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	initialize(initialize)
	retrieve(retrieve)
	grade_context(grade_context)
	rewrite_question(rewrite_question)
	generate_answer(generate_answer)
	fallback_answer(fallback_answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> initialize;
	grade_context -. &nbsp;fallback&nbsp; .-> fallback_answer;
	grade_context -. &nbsp;generate&nbsp; .-> generate_answer;
	grade_context -. &nbsp;rewrite&nbsp; .-> rewrite_question;
	initialize --> retrieve;
	retrieve --> grade_context;
	rewrite_question --> retrieve;
	fallback_answer --> __end__;
	generate_answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Run the RAG Agent

This question should retrieve Chopin documents and generate an answer.

In [16]:
result = rag_graph.invoke(
    {
        "question": "What makes Chopin's piano music distinctive?",
        "max_attempts": 2,
    }
)

print("ANSWER:")
print(result["answer"])
print()
print("TRACE:")
for item in result["trace"]:
    print("-", item)
print()
print("RETRIEVED SOURCES:")
for doc in result["retrieved_docs"]:
    print(f'- {doc["source"]}: score={doc["score"]:.3f}')

ANSWER:
Chopin's piano music is distinctive for its expressive qualities, lyrical melodies, and use of rubato, as well as its focus on forms such as nocturnes, mazurkas, polonaises, and etudes, which highlight his Romantic era style. These elements contribute to the unique emotional depth and technical innovation in his compositions [doc_1_chopin_style].

TRACE:
- initialize
- retrieve: "What makes Chopin's piano music distinctive?"
- grade_context: True (best_score=0.342, keyword_overlap=['music', 'piano'])
- generate_answer

RETRIEVED SOURCES:
- doc_1_chopin_style: score=0.342
- doc_3_bach: score=0.127
- doc_2_chopin_paris: score=0.118


## Stream the Graph

Streaming shows what each node returns.

This is one of the best ways to teach and debug LangGraph workflows.

In [17]:
for step in rag_graph.stream(
    {
        "question": "Where did Chopin spend much of his adult life?",
        "max_attempts": 2,
    }
):
    print(step)
    print()

{'initialize': {'original_question': 'Where did Chopin spend much of his adult life?', 'question': 'Where did Chopin spend much of his adult life?', 'attempts': 0, 'max_attempts': 2, 'trace': ['initialize']}}

{'retrieve': {'retrieved_docs': [{'source': 'doc_2_chopin_paris', 'text': 'Chopin spent much of his adult life in Paris, where he became connected with salon culture and many artists of the nineteenth century. His performances were often intimate rather than large public concerts.', 'score': 0.4115552142097566}, {'source': 'doc_1_chopin_style', 'text': 'Frederic Chopin was a Polish composer and pianist of the Romantic era. He is especially known for expressive piano music, lyrical melodies, rubato, nocturnes, mazurkas, polonaises, and etudes.', 'score': 0.0978202000545384}, {'source': 'doc_5_debussy', 'text': 'Claude Debussy was a French composer associated with impressionist colors, unusual scales, atmosphere, and works such as Prelude to the Afternoon of a Faun.', 'score': 0.0}

## Try a Weak Question

This question is outside the knowledge base.

The graph should retrieve weak context, rewrite once or twice, and then either answer from found context or fall back safely.

In [18]:
weak_result = rag_graph.invoke(
    {
        "question": "What is the capital city of Mars?",
        "max_attempts": 1,
    }
)

print("ANSWER:")
print(weak_result["answer"])
print()
print("TRACE:")
for item in weak_result["trace"]:
    print("-", item)

ANSWER:
I could not find enough relevant information in the local knowledge base to answer confidently.

TRACE:
- initialize
- retrieve: "What is the capital city of Mars?"
- grade_context: False (best_score=0.000, keyword_overlap=[])
- rewrite_question: attempt 1
- retrieve: "What is the capital city of Mars? classical music composer style biography works Romantic Baroque"
- grade_context: False (best_score=0.358, keyword_overlap=[])
- fallback_answer


## Compare With a Linear Chain

A basic chain would look like this:

```text
question -> retrieve -> generate -> answer
```

Our graph can do this:

```text
question -> retrieve -> grade
                         |
                         +-- good -> generate -> answer
                         |
                         +-- weak -> rewrite -> retrieve again
                         |
                         +-- still weak -> fallback
```

This is why LangGraph is a better fit here: the workflow is not just a straight line.

## Exercises

### Exercise 1: Add More Documents

Add two more composer documents to `DOCUMENTS`.

Try questions that should retrieve them.

### Exercise 2: Tune the Grader

Change the relevance threshold in `grade_context`:

```python
best_score >= 0.25
```

What happens if you make it stricter?

### Exercise 3: Add a Citation Formatter Node

Add a node after `generate_answer` that extracts source ids and formats a final citation list.

### Exercise 4: Replace the Retriever

Replace the TF-IDF retriever with:

- OpenAI embeddings
- FAISS
- Chroma
- PDF loading with `PyPDFLoader`

### Exercise 5: Add an LLM Rewriter

Change `rewrite_question` so it calls an LLM to rewrite the query.

### Exercise 6: Explain the Need for LangGraph

In your own words, explain why this RAG workflow is more naturally represented as a graph than as a simple chain.